# EDA
Import the sample.csv from Weights & Biases and perform basic cleanup

Note: the environment that was made was `mlflow-82b37777b465ed71625d480648d6f5e09b319f0d`

In [ ]:
import wandb
import pandas as pd

# Import data from Weights & Biases
run = wandb.init(project="nyc_airbnb", group="eda", save_code=True)
local_path = wandb.use_artifact("sample.csv:latest").file()
df = pd.read_csv(local_path)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/t4corun/.netrc.
wandb: Currently logged in as: t4corun (t4corun-mledp) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Artifact "source-nyc_airbnb-_home_t4corun_repo_build-ml-pipeline-for-short-term-rental-prices_src_eda_EDA.ipynb" already exists with the same content. No new version will be created.


# Profiling step
- there are lots of missing cells
- 20000 rows
- missing values in these cols
    - name
    - host_name
    - last_review
    - reviews_per_month
- price does have outliers such as 0 and 10000


questions/actions
- udacity says last_review is a string when it should be a date. the profiling says date
- udacity says the prices should be between 10 and 350
- can I fill in the missing host_name from the other rows with the same host_id?

In [2]:
import ydata_profiling

profile = ydata_profiling.ProfileReport(df)
profile.to_notebook_iframe()

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 16/16 [00:00<00:00, 18.02it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

# Basic fixes
- Drop the outlier prices
- convert the last_review to type datetime

In [3]:
# Drop outliers
min_price = 10
max_price = 350
idx = df['price'].between(min_price, max_price)
df = df[idx].copy()

# Convert last_review to datetime
df['last_review'] = pd.to_datetime(df['last_review'])

# check the data
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 19001 entries, 0 to 19999
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   id                              19001 non-null  int64         
 1   name                            18994 non-null  object        
 2   host_id                         19001 non-null  int64         
 3   host_name                       18993 non-null  object        
 4   neighbourhood_group             19001 non-null  object        
 5   neighbourhood                   19001 non-null  object        
 6   latitude                        19001 non-null  float64       
 7   longitude                       19001 non-null  float64       
 8   room_type                       19001 non-null  object        
 9   price                           19001 non-null  int64         
 10  minimum_nights                  19001 non-null  int64         
 11  number_

In [4]:
# finish the run
run.finish()